# Notebook 28
## Distance and Mathematical Similarity
Instance based algorithms rely on measuring the physical distance between data points.

### Euclidean Distance
Formula for the shortest straight-line distance between two points.

Scales up to calculate the distance across thousands of dimensitons simulatenously.

### Similarity Matching
Core assumption that points mathematically closer to each other in vector space are semantically similar. 

To find the most similar face to another, you calculate its Euclidean distance to every other face and find the minimum value.

### Intra vs. Inter Class Distance
Metric for how well your data is separated.

Intra-Class Distance
- Distance between two samples of the same category.
- Good dataset has very low intra-class distance.

Inter-Class Distance
- Distance between samples of different categories.
- Good dataset has very high inter class distance.

## K-Nearest Neighbors
Instance based learning algorithm. 

Unlike neural networks, this doesn't actually learn a formula during training.
- Just memorizes the dataset.

### How it Predicts
When a brand new data point arrives, this calculates the Euclidean distance between that new point and every single point in the dataset.

### K Parameter
Finds the k closest training points.

### Majority Vote
New data point is assigned, the class that has the majority vote among those k neighbors.

## High-Dimensional Data Handling
### Scaling Up Image Dimensions
When you  flatten 64x64 images, they become a single row of 4096 features.

This increase in features leads to distance calculations that become incredibly slow and computationaly expensive.

### Dimensionality Reduction
Principal Component Analysis is an algorithm used to solve high-dimensional problems. 

It compresses data, instead of keeping all 4096 pixels, PCA extracts the most critical structural patterns and throws away the noise.

## Applied Machine Learning Domains
### Facial Recognition
Represents a shift from basic pattern recognition to biometric identification.

It's treated as a massive multiclass classification problem where every single unique person is their own target class.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.decomposition import PCA

# ==========================================
# TOPIC 1: FACIAL RECOGNITION & SCALING UP DIMENSIONS
# ==========================================
print("--- 1. Facial Recognition & High-Dimensional Images ---")
# Load the dataset. Faces are 64x64 pixels.
faces = fetch_olivetti_faces(shuffle=True, random_state=42)
X = faces.data        # Flattened vectors
y = faces.target      # Target labels (Person ID 0 through 39)
images = faces.images # 2D image matrices

print(f"Loaded {X.shape[0]} images of {len(np.unique(y))} different people.")
print(f"Original image shape: {images.shape[1]}x{images.shape[2]} pixels")
print(f"Flattened 1D vector shape (The Features): {X.shape[1]} dimensions!\n")


# ==========================================
# TOPIC 2: INTRA-CLASS VS INTER-CLASS DISTANCE
# ==========================================
print("--- 2. Intra-Class vs. Inter-Class Distance ---")
# Find two images of the SAME person (Person 0)
same_person_idx = np.where(y == 0)[0]
img_A_idx = same_person_idx[0]
img_B_idx = same_person_idx[1]

# Find an image of a DIFFERENT person (Person 1)
diff_person_idx = np.where(y == 1)[0][0]

# Calculate Euclidean Distances
# (We pass them as 2D arrays because scikit-learn expects lists of samples)
dist_intra = euclidean_distances([X[img_A_idx]], [X[img_B_idx]])[0][0]
dist_inter = euclidean_distances([X[img_A_idx]], [X[diff_person_idx]])[0][0]

print(f"Intra-class distance (Person 0 to Person 0): {dist_intra:.2f}")
print(f"Inter-class distance (Person 0 to Person 1): {dist_inter:.2f}")
print("Notice how images of the same person are mathematically closer together in space.\n")


# ==========================================
# TOPIC 3: EUCLIDEAN DISTANCE & SIMILARITY MATCHING
# ==========================================
print("--- 3. Euclidean Similarity Matching ---")
query_idx = 42 # Arbitrary image to act as our "query"
query_vector = X[query_idx]

# Calculate the distance from our query to EVERY OTHER image in the dataset
all_distances = euclidean_distances([query_vector], X)[0]

# Set the distance to itself to infinity so it doesn't match with itself
all_distances[query_idx] = np.inf 

# Find the index of the absolute minimum distance
closest_match_idx = np.argmin(all_distances)

print(f"Query Image is Person ID: {y[query_idx]}")
print(f"The mathematically closest image is Person ID: {y[closest_match_idx]}")
print(f"Distance between them: {all_distances[closest_match_idx]:.2f}")

# Plot them side-by-side
fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(images[query_idx], cmap="gray")
axes[0].set_title(f"Query: Person {y[query_idx]}")
axes[0].axis("off")
axes[1].imshow(images[closest_match_idx], cmap="gray")
axes[1].set_title(f"Closest Match: Person {y[closest_match_idx]}")
axes[1].axis("off")
plt.suptitle("Similarity Matching via Euclidean Distance")
plt.show() # NOTE: Close the plot window to continue!
print()


# ==========================================
# TOPIC 4: K-NEAREST NEIGHBORS (KNN)
# ==========================================
print("--- 4. K-Nearest Neighbors Classification ---")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

print("Training KNN (k=3) on raw 4096-dimensional data...")
start_time = time.time()

knn_raw = KNeighborsClassifier(n_neighbors=3)
knn_raw.fit(X_train, y_train)
y_pred_raw = knn_raw.predict(X_test)
acc_raw = accuracy_score(y_test, y_pred_raw)

raw_time = time.time() - start_time
print(f"Raw Accuracy: {acc_raw*100:.1f}%")
print(f"Time taken to predict: {raw_time:.4f} seconds\n")


# ==========================================
# TOPIC 5: DIMENSIONALITY REDUCTION (PCA)
# ==========================================
print("--- 5. Dimensionality Reduction (PCA) ---")
print("4,096 dimensions is too many. Let's compress the images down to just 50 core features.")

# Initialize PCA to keep 50 components
pca = PCA(n_components=50, random_state=42)

# We "fit" PCA on the training data to learn the core facial structures, 
# then "transform" both train and test sets into the new compressed 50D space.
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"Old training shape: {X_train.shape}")
print(f"New compressed PCA training shape: {X_train_pca.shape}\n")

print("Training a NEW KNN (k=3) on the compressed 50-dimensional data...")
start_time = time.time()

knn_pca = KNeighborsClassifier(n_neighbors=3)
knn_pca.fit(X_train_pca, y_train)
y_pred_pca = knn_pca.predict(X_test_pca)
acc_pca = accuracy_score(y_test, y_pred_pca)

pca_time = time.time() - start_time
print(f"PCA Accuracy: {acc_pca*100:.1f}%")
print(f"Time taken to predict: {pca_time:.4f} seconds")
print("\nNotice how PCA massively speeds up the Euclidean distance calculations without losing much (if any) accuracy!")